# QuantumX Transfinite-1 — Hybrid Quantum ECG Classifier (Claude Edition)

**Prerequisite**: Run `01_Classical_Training.ipynb` first (generates encoder + benchmark report)  
**Architecture**: Frozen ResNet-18 → FC(512→8) → 8-Qubit VQC (StronglyEntanglingLayers) → FC(8→4)  
**Output**: `artifacts_v1/` (VQC weights, model) + `benchmarks/` (quantum graphs + comparison)

In [ ]:
import os, sys, json, time, copy, warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import models
import pennylane as qml

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_fscore_support, matthews_corrcoef, f1_score
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.2)
print(f'PennyLane: {qml.__version__}')
print('Imports OK')

## 1. Paths & Device

In [ ]:
MODEL_ROOT = Path(os.getcwd()).resolve()
# MODEL_ROOT = Path(r'C:\Users\anshu\OneDrive\Desktop\QuantumX\Models\v1 - Heart Attack (ECG Image) - Claude')

ARTIFACTS = MODEL_ROOT / 'artifacts_v1'
GRAPHS = MODEL_ROOT / 'benchmarks'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
GRAPHS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(MODEL_ROOT / 'core'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Model Root: {MODEL_ROOT}')
print(f'Device: {device} ({gpu_name})')

## 2. Load Dataset

In [ ]:
from ecg_image_dataset import (
    create_cardiac_dataloaders, CardiacECGImageDataset,
    CLASS_NAMES, NUM_CLASSES, IMAGENET_MEAN, IMAGENET_STD,
    get_val_transforms, get_train_transforms, find_dataset_root, HOLDOUT_DIR
)

data = create_cardiac_dataloaders(
    batch_size=16, image_size=224, num_workers=2,
    holdout_per_class=30, val_split=0.15, seed=42
)

train_loader = data['train_loader']
val_loader = data['val_loader']
test_loader = data['test_loader']
class_weights = data['class_weights'].to(device)
summary = data['dataset_summary']
dataset_root = data['dataset_root']

print(f"Train: {summary['train_samples']} | Val: {summary['val_samples']} | Test: {summary['test_samples']}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

## 3. Quantum Circuit Definition

In [ ]:
N_QUBITS = 8
N_LAYERS = 2

qdev = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(qdev, interface='torch', diff_method='backprop')
def quantum_circuit(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation='Y')
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_QUBITS)
n_quantum_params = int(np.prod(weight_shape))

print(f'Qubits: {N_QUBITS}')
print(f'Layers: {N_LAYERS}')
print(f'Ansatz: StronglyEntanglingLayers')
print(f'Encoding: AngleEmbedding (RY)')
print(f'Measurement: PauliZ on all qubits')
print(f'Quantum Parameters: {n_quantum_params}')
print(f'Weight shape: {weight_shape}')

## 4. Circuit Diagram

In [ ]:
dummy_inputs = torch.zeros(N_QUBITS)
dummy_weights = torch.randn(*weight_shape)

fig, ax = qml.draw_mpl(quantum_circuit, style='pennylane')(dummy_inputs, dummy_weights)
fig.set_size_inches(20, 8)
fig.suptitle('Transfinite-1: 8-Qubit VQC Architecture', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(GRAPHS / 'quantum_circuit_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

# Text version
circuit_text = qml.draw(quantum_circuit)(dummy_inputs, dummy_weights)
print(circuit_text)
with open(GRAPHS / 'quantum_circuit_text.txt', 'w', encoding='utf-8') as f:
    f.write(circuit_text)

## 5. Build Hybrid Model

In [ ]:
class HybridQuantumCardiac(nn.Module):
    def __init__(self, num_classes=4, n_qubits=8, n_layers=2):
        super().__init__()
        self.n_qubits = n_qubits
        
        # Frozen ResNet-18 encoder
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        for p in self.encoder.parameters():
            p.requires_grad = False
        
        # Classical bottleneck: 512 → 8 (for quantum embedding)
        self.bottleneck = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 64), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(64, n_qubits), nn.Tanh()  # [-1, 1] for angle embedding
        )
        
        # Quantum weights
        ws = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)
        self.qweights = nn.Parameter(torch.randn(*ws) * 0.1)
        
        # Classical readout: 8 → num_classes
        self.readout = nn.Sequential(
            nn.Linear(n_qubits, 32), nn.ReLU(True), nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        with torch.no_grad():
            features = self.encoder(x)
        bottleneck_out = self.bottleneck(features)
        quantum_out = []
        for i in range(bottleneck_out.shape[0]):
            q_result = torch.stack(quantum_circuit(bottleneck_out[i], self.qweights))
            quantum_out.append(q_result)
        return self.readout(torch.stack(quantum_out))

hybrid = HybridQuantumCardiac(NUM_CLASSES, N_QUBITS, N_LAYERS).to(device)

total_p = sum(p.numel() for p in hybrid.parameters())
trainable_p = sum(p.numel() for p in hybrid.parameters() if p.requires_grad)
frozen_p = total_p - trainable_p

print(f'\nHybrid Quantum-Classical Architecture:')
print(f'  Total params     : {total_p:>10,}')
print(f'  Trainable params : {trainable_p:>10,}')
print(f'  Frozen (encoder) : {frozen_p:>10,}')
print(f'  Quantum params   : {n_quantum_params:>10,}')

## 6. Train — Every Epoch Logged Individually

In [ ]:
EPOCHS_Q = 30
LR_Q = 5e-3
PATIENCE_Q = 10

criterion = nn.CrossEntropyLoss(weight=class_weights)
opt = optim.Adam(filter(lambda p: p.requires_grad, hybrid.parameters()), lr=LR_Q, weight_decay=1e-4)
scheduler = CosineAnnealingLR(opt, T_max=EPOCHS_Q, eta_min=1e-6)

hq = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': [], 'epoch_time': []}
best_acc_q = 0.0
best_state_q = None
patience_cnt = 0
total_start = time.time()

print('=' * 110)
print('  TRAINING TRANSFINITE-1 HYBRID QUANTUM MODEL — 8-Qubit VQC on ECG Images')
print('=' * 110)
print(f'{"Epoch":>7} | {"TrLoss":>8} {"TrAcc":>7} | {"VaLoss":>8} {"VaAcc":>7} | {"LR":>10} | {"EpochT":>8} | {"TotalT":>8} | Status')
print('-' * 110)

for epoch in range(1, EPOCHS_Q + 1):
    epoch_start = time.time()
    
    # ── TRAIN ──
    hybrid.train()
    running_loss, correct, total = 0.0, 0, 0
    n_batches = len(train_loader)
    
    for batch_idx, (images, labels, _) in enumerate(train_loader, 1):
        images, labels = images.to(device), labels.to(device)
        opt.zero_grad()
        outputs = hybrid(images)
        loss = criterion(outputs, labels)
        loss.backward()
        opt.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # Per-batch progress
        if batch_idx % 3 == 0 or batch_idx == n_batches:
            batch_acc = 100.0 * correct / total
            elapsed = time.time() - epoch_start
            print(f'\r  Epoch {epoch:02d} | Batch {batch_idx:>4d}/{n_batches} | Loss: {loss.item():.4f} | Acc: {batch_acc:.1f}% | {elapsed:.0f}s', end='', flush=True)
    
    train_loss = running_loss / total
    train_acc = 100.0 * correct / total
    
    # ── VALIDATE ──
    hybrid.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels, _ in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = hybrid(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_loss = val_loss / val_total
    val_acc = 100.0 * val_correct / val_total
    
    scheduler.step()
    lr = opt.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start
    total_elapsed = time.time() - total_start
    
    hq['train_loss'].append(train_loss)
    hq['val_loss'].append(val_loss)
    hq['train_acc'].append(train_acc)
    hq['val_acc'].append(val_acc)
    hq['lr'].append(lr)
    hq['epoch_time'].append(epoch_time)
    
    status = ''
    if val_acc > best_acc_q:
        best_acc_q = val_acc
        best_state_q = copy.deepcopy(hybrid.state_dict())
        patience_cnt = 0
        status = '★ BEST'
    else:
        patience_cnt += 1
        status = f'  (patience {patience_cnt}/{PATIENCE_Q})'
    
    et = str(timedelta(seconds=int(epoch_time)))
    tt = str(timedelta(seconds=int(total_elapsed)))
    print(f'\r  {epoch:02d}/{EPOCHS_Q:02d}  | {train_loss:>8.4f} {train_acc:>6.1f}% | {val_loss:>8.4f} {val_acc:>6.1f}% | {lr:>10.6f} | {et:>8} | {tt:>8} | {status}')
    
    if patience_cnt >= PATIENCE_Q:
        print(f'\n  ⚡ Early stopping at epoch {epoch}. Best val acc: {best_acc_q:.2f}%')
        break

total_time_q = time.time() - total_start
print('-' * 110)
print(f'  Training complete in {str(timedelta(seconds=int(total_time_q)))}')
print(f'  Best validation accuracy: {best_acc_q:.2f}%')
print(f'  Average epoch time: {np.mean(hq["epoch_time"]):.1f}s')

hybrid.load_state_dict(best_state_q)

## 7. Save Model & VQC Weights

In [ ]:
np.save(ARTIFACTS / 'cardiac_vqc_weights.npy', hybrid.qweights.detach().cpu().numpy())

torch.save({
    'state_dict': best_state_q,
    'n_qubits': N_QUBITS,
    'n_layers': N_LAYERS,
    'num_classes': NUM_CLASSES,
    'best_val_acc': best_acc_q,
    'quantum_params': n_quantum_params,
    'trainable_params': trainable_p,
    'training_time_s': total_time_q,
}, ARTIFACTS / 'cardiac_hybrid_production.pt')

print(f'Saved: {ARTIFACTS / "cardiac_vqc_weights.npy"}')
print(f'Saved: {ARTIFACTS / "cardiac_hybrid_production.pt"}')

## 8. Test Evaluation

In [ ]:
hybrid.eval()
test_loss_q, test_correct_q, test_total_q = 0.0, 0, 0
all_preds_q, all_labels_q, all_probs_q = [], [], []

with torch.no_grad():
    for images, labels, _ in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = hybrid(images)
        loss = criterion(outputs, labels)
        test_loss_q += loss.item() * images.size(0)
        probs = torch.softmax(outputs, 1)
        _, predicted = outputs.max(1)
        test_total_q += labels.size(0)
        test_correct_q += predicted.eq(labels).sum().item()
        all_preds_q.extend(predicted.cpu().numpy())
        all_labels_q.extend(labels.cpu().numpy())
        all_probs_q.extend(probs.cpu().numpy())

test_loss_q = test_loss_q / test_total_q
test_acc_q = 100.0 * test_correct_q / test_total_q
preds_q = np.array(all_preds_q)
labels_q = np.array(all_labels_q)
probs_q = np.array(all_probs_q)

mcc_q = matthews_corrcoef(labels_q, preds_q)
prec_q, rec_q, f1q, _ = precision_recall_fscore_support(labels_q, preds_q, average=None)
macro_f1_q = float(f1_score(labels_q, preds_q, average='macro'))

print('=' * 70)
print('  TEST RESULTS — Transfinite-1 Hybrid Quantum')
print('=' * 70)
print(f'  Test Accuracy : {test_acc_q:.2f}%')
print(f'  Test Loss     : {test_loss_q:.6f}')
print(f'  Macro F1      : {macro_f1_q:.4f}')
print(f'  MCC           : {mcc_q:.4f}')
print()
print(classification_report(labels_q, preds_q, target_names=CLASS_NAMES, digits=4))

## 9. Quantum Graphs

In [ ]:
eps = range(1, len(hq['train_loss']) + 1)

# 9.1 Loss
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(eps, hq['train_loss'], color='purple', lw=2, marker='o', ms=4, label='Train')
ax.plot(eps, hq['val_loss'], color='darkorange', lw=2, marker='s', ms=4, label='Validation')
ax.set(xlabel='Epoch', ylabel='Loss', title='Transfinite-1: Training & Validation Loss')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(GRAPHS / 'quantum_loss_curves.png', dpi=150); plt.show()

# 9.2 Accuracy
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(eps, hq['train_acc'], color='purple', lw=2, marker='o', ms=4, label='Train')
ax.plot(eps, hq['val_acc'], color='darkorange', lw=2, marker='s', ms=4, label='Validation')
ax.axhline(best_acc_q, color='g', ls='--', alpha=0.7, label=f'Best: {best_acc_q:.1f}%')
ax.set(xlabel='Epoch', ylabel='Accuracy (%)', title='Transfinite-1: Training & Validation Accuracy')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(GRAPHS / 'quantum_accuracy_curves.png', dpi=150); plt.show()

# 9.3 Epoch Time
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(eps, hq['epoch_time'], color='#AB47BC', alpha=0.8)
ax.axhline(np.mean(hq['epoch_time']), color='r', ls='--', label=f'Avg: {np.mean(hq["epoch_time"]):.1f}s')
ax.set(xlabel='Epoch', ylabel='Time (s)', title='Transfinite-1: Time per Epoch')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(GRAPHS / 'quantum_epoch_times.png', dpi=150); plt.show()

print(f'Saved 3 training graphs')

In [ ]:
# 9.4 Confusion Matrix
cm_q = confusion_matrix(labels_q, preds_q)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_q, annot=True, fmt='d', cmap='Purples', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, annot_kws={'size': 14, 'weight': 'bold'}, linewidths=0.5)
ax.set(xlabel='Predicted', ylabel='True', title='Transfinite-1: Confusion Matrix')
plt.tight_layout(); plt.savefig(GRAPHS / 'quantum_confusion_matrix.png', dpi=150); plt.show()

# 9.5 ROC
lbin = label_binarize(labels_q, classes=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#9C27B0', '#E91E63', '#FF5722', '#009688']
aucs_q = {}
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(lbin[:, i], probs_q[:, i])
    a = auc(fpr, tpr); aucs_q[CLASS_NAMES[i]] = float(a)
    ax.plot(fpr, tpr, color=colors[i], lw=2, label=f'{CLASS_NAMES[i]} (AUC={a:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax.set(xlabel='FPR', ylabel='TPR', title='Transfinite-1: ROC Curves')
ax.legend(fontsize=11, loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(GRAPHS / 'quantum_roc_curves.png', dpi=150); plt.show()

macro_auc_q = float(np.mean(list(aucs_q.values())))
print(f'Macro AUC: {macro_auc_q:.4f}')

## 10. Classical vs Quantum Comparison

In [ ]:
# Load classical report
cls_report_path = ARTIFACTS / 'cardiac_benchmark_report.json'
if cls_report_path.exists():
    with open(cls_report_path) as f:
        cls_report = json.load(f)
    ca = cls_report['metrics']['test_accuracy']
    cf = cls_report['metrics']['macro_f1']
    cm_cls = cls_report['metrics']['mcc']
    ca_auc = cls_report['metrics']['macro_auc']
    cls_params = cls_report['parameters']['trainable']
    print(f'Classical loaded: Acc={ca:.2f}% F1={cf:.4f} MCC={cm_cls:.4f} AUC={ca_auc:.4f}')
else:
    ca, cf, cm_cls, ca_auc, cls_params = 0, 0, 0, 0, 11_200_000
    print('WARNING: No classical report found. Run 01_Classical_Training first.')

# Comparison bar chart
metric_names = ['Accuracy', 'Macro F1', 'MCC', 'Macro AUC']
cls_vals = [ca/100 if ca > 1 else ca, cf, cm_cls, ca_auc]
q_vals = [test_acc_q/100, macro_f1_q, float(mcc_q), macro_auc_q]

x = np.arange(len(metric_names)); w = 0.35
fig, ax = plt.subplots(figsize=(14, 7))
b1 = ax.bar(x - w/2, cls_vals, w, label='CX-01 Classical', color='#2196F3')
b2 = ax.bar(x + w/2, q_vals, w, label='Transfinite-1 Quantum', color='#9C27B0')
ax.set(xticks=x, ylabel='Score', title='CX-01 Classical vs Transfinite-1 Quantum: Head-to-Head')
ax.set_xticklabels(metric_names, fontsize=12)
ax.legend(fontsize=12); ax.set_ylim(0, 1.15); ax.grid(alpha=0.3, axis='y')
for bars in [b1, b2]:
    for b in bars:
        ax.annotate(f'{b.get_height():.3f}', (b.get_x() + b.get_width()/2, b.get_height()),
                    ha='center', fontsize=11, fontweight='bold', textcoords='offset points', xytext=(0, 4))
plt.tight_layout(); plt.savefig(GRAPHS / 'classical_vs_quantum_comparison.png', dpi=150); plt.show()

# Parameter efficiency
fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 6))
a1.bar(['CX-01\nClassical', 'Transfinite-1\nQuantum'], [cls_params, trainable_p],
       color=['#2196F3', '#9C27B0'], width=0.5)
a1.set_ylabel('Trainable Parameters'); a1.set_title('Parameter Count Comparison')
for i, v in enumerate([cls_params, trainable_p]):
    a1.text(i, v + max(cls_params, trainable_p)*0.02, f'{v:,}', ha='center', fontweight='bold')
a1.grid(alpha=0.3, axis='y')

enc_p = sum(p.numel() for p in hybrid.encoder.parameters())
bn_p = sum(p.numel() for p in hybrid.bottleneck.parameters())
ro_p = sum(p.numel() for p in hybrid.readout.parameters())
a2.pie([enc_p, bn_p, n_quantum_params, ro_p],
       labels=['Frozen Encoder', 'Bottleneck', f'VQC ({n_quantum_params})', 'Readout'],
       colors=['#E0E0E0', '#64B5F6', '#CE93D8', '#81C784'],
       autopct='%1.1f%%', explode=(0, 0, 0.1, 0), shadow=True)
a2.set_title('Transfinite-1: Parameter Breakdown')
plt.tight_layout(); plt.savefig(GRAPHS / 'parameter_efficiency_comparison.png', dpi=150); plt.show()

## 11. Scarce-Data Quantum Advantage Evaluation

In [ ]:
fractions = [0.10, 0.15, 0.25, 0.50, 1.0]
scarce_classical = []
scarce_quantum = []

print('Scarce-Data Evaluation (this takes time)...')
print(f'{"Fraction":>10} | {"Classical":>10} | {"Quantum":>10} | {"Δ":>8}')
print('-' * 45)

for frac in fractions:
    full_dataset = CardiacECGImageDataset(str(dataset_root), 'train', get_train_transforms(224))
    n_samples = max(int(len(full_dataset) * frac), 20)
    torch.manual_seed(42)
    indices = torch.randperm(len(full_dataset))[:n_samples].tolist()
    subset = torch.utils.data.Subset(full_dataset, indices)
    sub_loader = torch.utils.data.DataLoader(subset, 16, shuffle=True, num_workers=0, drop_last=True)
    
    # Quick classical
    cm_sc = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    cm_sc.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, NUM_CLASSES))
    cm_sc = cm_sc.to(device)
    opt_sc = optim.Adam(cm_sc.parameters(), lr=1e-3)
    cc = nn.CrossEntropyLoss(weight=class_weights)
    for _ in range(10):
        cm_sc.train()
        for im, lb, _ in sub_loader:
            im, lb = im.to(device), lb.to(device)
            opt_sc.zero_grad(); cc(cm_sc(im), lb).backward(); opt_sc.step()
    cm_sc.eval()
    c_correct, c_total = 0, 0
    with torch.no_grad():
        for im, lb, _ in test_loader:
            im, lb = im.to(device), lb.to(device)
            _, p = cm_sc(im).max(1); c_total += lb.size(0); c_correct += p.eq(lb).sum().item()
    c_acc = 100.0 * c_correct / c_total
    scarce_classical.append(c_acc)
    
    # Quick quantum
    qm_sc = HybridQuantumCardiac(NUM_CLASSES, N_QUBITS, N_LAYERS).to(device)
    opt_qsc = optim.Adam(filter(lambda p: p.requires_grad, qm_sc.parameters()), lr=5e-3)
    for _ in range(10):
        qm_sc.train()
        for bi, (im, lb, _) in enumerate(sub_loader):
            if bi >= 5: break
            im, lb = im.to(device), lb.to(device)
            opt_qsc.zero_grad(); cc(qm_sc(im), lb).backward(); opt_qsc.step()
    qm_sc.eval()
    q_correct, q_total = 0, 0
    with torch.no_grad():
        for im, lb, _ in test_loader:
            im, lb = im.to(device), lb.to(device)
            _, p = qm_sc(im).max(1); q_total += lb.size(0); q_correct += p.eq(lb).sum().item()
    q_acc = 100.0 * q_correct / q_total
    scarce_quantum.append(q_acc)
    
    delta = q_acc - c_acc
    print(f'{frac*100:>9.0f}% | {c_acc:>9.1f}% | {q_acc:>9.1f}% | {delta:>+7.1f}%')
    
    del cm_sc, qm_sc; torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Plot
fig, ax = plt.subplots(figsize=(12, 7))
pcts = [f * 100 for f in fractions]
ax.plot(pcts, scarce_classical, 'b-o', lw=2.5, ms=10, label='CX-01 Classical')
ax.plot(pcts, scarce_quantum, color='purple', lw=2.5, marker='D', ms=10, label='Transfinite-1 Quantum')
for i in range(len(pcts)):
    if scarce_quantum[i] > scarce_classical[i]:
        ax.annotate(f'+{scarce_quantum[i]-scarce_classical[i]:.1f}%', (pcts[i], scarce_quantum[i]),
                    xytext=(10, 10), textcoords='offset points', fontsize=10, color='green', fontweight='bold')
ax.set(xlabel='Training Data Used (%)', ylabel='Test Accuracy (%)', title='Scarce-Cohort Quantum Resilience Analysis')
ax.legend(fontsize=12); ax.grid(alpha=0.3); ax.set_xticks(pcts)
plt.tight_layout(); plt.savefig(GRAPHS / 'scarce_data_comparison.png', dpi=150); plt.show()

## 12. Quantum Gate Saliency Heatmap

In [ ]:
hybrid.eval()
hybrid.qweights.requires_grad_(True)
total_grad = torch.zeros_like(hybrid.qweights)
count = 0

for images, labels, _ in test_loader:
    if count >= 30: break
    images = images.to(device)
    for i in range(min(images.size(0), 30 - count)):
        hybrid.zero_grad()
        out = hybrid(images[i:i+1])
        out[0, out.argmax(1)].backward(retain_graph=True)
        if hybrid.qweights.grad is not None:
            total_grad += hybrid.qweights.grad.abs()
            hybrid.qweights.grad.zero_()
        count += 1

saliency = (total_grad / max(count, 1)).detach().cpu().numpy()

fig, axes = plt.subplots(1, N_LAYERS, figsize=(8 * N_LAYERS, 6))
if N_LAYERS == 1: axes = [axes]
for li in range(N_LAYERS):
    sns.heatmap(saliency[li], annot=True, fmt='.3f', cmap='magma',
                xticklabels=['Rot-X', 'Rot-Y', 'Rot-Z'],
                yticklabels=[f'Q{i}' for i in range(N_QUBITS)],
                ax=axes[li], linewidths=0.5, annot_kws={'size': 10})
    axes[li].set_title(f'Layer {li+1} Gate Saliency', fontsize=14, fontweight='bold')
plt.suptitle('Transfinite-1: Quantum Gate Importance Analysis', fontsize=16, fontweight='bold')
plt.tight_layout(); plt.savefig(GRAPHS / 'quantum_gate_saliency.png', dpi=150); plt.show()

## 13. Save Final Report

In [ ]:
quantum_report = {
    'model_name': 'Transfinite-1 Cardiac Quantum',
    'model_type': 'hybrid_quantum',
    'architecture': 'Frozen ResNet-18 → FC(512→8) → 8-Qubit VQC → FC(8→4)',
    'quantum': {
        'qubits': N_QUBITS, 'layers': N_LAYERS,
        'ansatz': 'StronglyEntanglingLayers',
        'encoding': 'AngleEmbedding(RY)',
        'measurement': 'PauliZ',
        'params': n_quantum_params,
    },
    'training': {
        'epochs': len(hq['train_loss']),
        'time_s': round(total_time_q, 1),
        'lr': LR_Q,
        'gpu': gpu_name,
    },
    'parameters': {'total': total_p, 'trainable': trainable_p, 'frozen': frozen_p, 'quantum': n_quantum_params},
    'metrics': {
        'test_accuracy': round(test_acc_q, 4),
        'test_loss': round(test_loss_q, 6),
        'macro_f1': round(macro_f1_q, 4),
        'mcc': round(float(mcc_q), 4),
        'macro_auc': round(macro_auc_q, 4),
        'per_class_auc': {k: round(v, 4) for k, v in aucs_q.items()},
        'per_class_f1': {CLASS_NAMES[i]: round(float(f1q[i]), 4) for i in range(NUM_CLASSES)},
        'confusion_matrix': cm_q.tolist(),
        'best_val_acc': round(best_acc_q, 4),
    },
    'scarce_data': {'fractions': fractions, 'classical_acc': scarce_classical, 'quantum_acc': scarce_quantum},
    'history': hq,
    'graphs': sorted([f.name for f in GRAPHS.glob('quantum_*.png')] + 
                     ['classical_vs_quantum_comparison.png', 'parameter_efficiency_comparison.png', 'scarce_data_comparison.png']),
    'generated_at': datetime.now().isoformat(),
}

# Merge into main report
report_path = ARTIFACTS / 'cardiac_benchmark_report.json'
if report_path.exists():
    with open(report_path) as f:
        main_report = json.load(f)
    main_report['quantum_model'] = quantum_report
else:
    main_report = {'quantum_model': quantum_report}

with open(report_path, 'w') as f:
    json.dump(main_report, f, indent=2)

print(f'\n{"=" * 70}')
print(f'  TRANSFINITE-1 HYBRID QUANTUM PIPELINE COMPLETE')
print(f'  Accuracy    : {test_acc_q:.2f}%')
print(f'  Macro F1    : {macro_f1_q:.4f}')
print(f'  Macro AUC   : {macro_auc_q:.4f}')
print(f'  MCC         : {mcc_q:.4f}')
print(f'  Q-Params    : {n_quantum_params}')
print(f'  Time        : {str(timedelta(seconds=int(total_time_q)))}')
print(f'  Report      : {report_path}')
print(f'  All Graphs  : {GRAPHS}')
print(f'{"=" * 70}')